In [1]:
import os
import numpy as np
import gymnasium
import highway_env
import warnings
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import EvalCallback

env_name = 'roundabout-v0'
root = f'{env_name}-PPO'

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ── Configuración ────────────────────────────────────────────────────────────
config = {
    "observation": {
        "type": "Kinematics",
        "vehicles_count": 8,
        "features": ["presence", "x", "y", "vx", "vy"],
        "normalize": True,
        "absolute": False,
    },
    "action": {
        "type": "DiscreteMetaAction",
    },
    "simulation_frequency": 15,
    "policy_frequency": 3,
    "show_trajectories": False
}

log_dir = f"{root}/logs/"
os.makedirs(log_dir, exist_ok=True)

# ── Entornos ─────────────────────────────────────────────────────────────────
def make_env():
    env = gymnasium.make(env_name)
    env.unwrapped.configure(config)
    env.reset()
    env = Monitor(env, log_dir)
    return env

env      = DummyVecEnv([make_env])
eval_env = DummyVecEnv([make_env])

# ── Callback ──────────────────────────────────────────────────────────────────
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=log_dir,
    log_path=log_dir,
    eval_freq=5_000,
    n_eval_episodes=5,
    deterministic=True,
    verbose=0,
)

# ── Modelo ────────────────────────────────────────────────────────────────────
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=5e-4,
)

# ── Entrenamiento ─────────────────────────────────────────────────────────────
print("Entrenando al agente...")
model.learn(total_timesteps=500_000, callback=eval_callback)

# Guardar el modelo final (además del mejor que ya guardó el callback)
model.save(f"{log_dir}/last_model")

env.close()
eval_env.close()

<frozen importlib._bootstrap>:488: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


Using cpu device
Entrenando al agente...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 20.7     |
|    ep_rew_mean     | 16.4     |
| time/              |          |
|    fps             | 19       |
|    iterations      | 1        |
|    time_elapsed    | 107      |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 21.7        |
|    ep_rew_mean          | 17.3        |
| time/                   |             |
|    fps                  | 17          |
|    iterations           | 2           |
|    time_elapsed         | 236         |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.012317147 |
|    clip_fraction        | 0.116       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.6        |
|    explained_variance   | 0.0